<a href="https://colab.research.google.com/github/jaydenchoe/python-lecture-jumptophython-examples/blob/main/gemini_api_chapter_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖<br>Gemini 챗봇 '제미' 조종하기 (고급편)

안녕하세요! 챗봇 '제미'를 만드는 데 성공하셨군요!

지난 시간에는 `client.models.generate_content()`와 `client.chats.create()`으로 '제미'와 대화를 나누는 법을 배웠습니다.

이번 시간에는 '제미'를 **우리의 의도대로 조종하는** 방법을 배울 거예요. '제미'의 창의력, 답변 길이, 심지어 안전장치까지 제어해서 '나만의 맞춤형 챗봇'을 만들어 봅시다!

(참고: 이 자료는 2025년 11월, 최신 `google-genai` SDK와 `gemini-2.5-flash` 모델을 기준으로 합니다.)

## 1단계: 챗봇을 위한 '두뇌' 설치 및 '열쇠' 등록하기 🧠🔑

가장 먼저, 최신 구글 Gen AI 라이브러리(`google-genai`)를 설치하고, 코랩의 '비밀' 기능(🔑)에 `GOOGLE_API_KEY`를 등록해야 해요.

**주의:** 예전의 `google-generativeai` 패키지는 이제 사용하지 않습니다!

In [ ]:
!pip install google-genai

In [ ]:
from google import genai
from google.genai import types
import os

# 🤫 코랩의 '비밀' 기능(userdata)에서 API 키를 가져옵니다.
try:
    from google.colab import userdata
    API_KEY = userdata.get('GOOGLE_API_KEY')

    # 클라이언트(Client)를 생성합니다. 이제 이 친구가 모든 것을 관리해요!
    client = genai.Client(api_key=API_KEY)
    print("✨ '제미'가 깨어날 준비가 되었어요! (최신 SDK 클라이언트 연결 완료)")
except Exception as e:
    print(f"🔑 비밀 키를 가져오는 데 실패했어요. {e}")
    print("--- (경고) --- \n왼쪽 🔑 아이콘에서 'GOOGLE_API_KEY'라는 이름으로 비밀 키를 꼭 저장해주세요!")

## 2단계: 챗봇의 '설정값' 만지기 (GenerateContentConfig)

'제미'의 모든 행동은 **`types.GenerateContentConfig`** 라는 '설정 객체'를 통해 제어할 수 있어요. (구버전의 `GenerationConfig`가 이름이 바뀌었어요!)

이 설정값은 '제미'가 대답을 *만들어내는(Generate)* 시점에 적용됩니다. '제미'의 창의력, 답변 길이, 금지어 등을 모두 여기서 설정해요.

In [ ]:
# 1. '설정 객체'를 만듭니다.
# (아직은 아무 설정도 안 했어요)
my_config = types.GenerateContentConfig()

# 2. 모델에게 부탁할 때(generate_content), 이 설정을 'config'로 함께 전달합니다.
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents="인공지능에 대해 1줄로 설명해줘.",
    config=my_config
)

print(response.text)

## 3단계: 🌡️ '제미'의 창의력 조절하기 (temperature)

가장 재미있는 설정입니다! **`temperature`**는 '제미'의 창의력(무작위성)을 조절해요.

* **`temperature = 0.0`**: 창의력 0. '제미'는 가장 확률 높은, 정해진 대답만 합니다. (매번 똑같은 대답을 해요!)
* **`temperature = 1.0`** (기본값): 창의력 풍부. '제미'가 다양한 단어를 조합해 매번 다른, 재미있는 대답을 해요.
* **`temperature = 2.0`** (최대값): 창의력 폭발! '제미'가 아주 엉뚱하고 이상한(...) 말을 할 수도 있어요.

창의력 0인 '제미'와 창의력 1인 '제미'를 비교해 볼까요?

In [ ]:
user_message = "겨울에 대한 짧은 시를 20자 이내로 지으세요."

print("--- 🌡️ Temperature = 0.0 (창의력 0) ---")
config_zero = types.GenerateContentConfig(temperature=0.0)

for _ in range(3): # 3번 반복 실행
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=user_message,
        config=config_zero
    )
    print("🤖:", response.text.strip())

print("\n--- 🌡️ Temperature = 1.0 (창의력 100) ---")
config_creative = types.GenerateContentConfig(temperature=1.0)

for _ in range(3): # 3번 반복 실행
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=user_message,
        config=config_creative
    )
    print("🤖:", response.text.strip())

## 4단계: ✂️ '제미' 말 끊기 (max_output_tokens)

'제미'가 너무 길게 말하면 요금이 많이 나올 수 있겠죠? **`max_output_tokens`**를 설정해서 '제미'의 답변을 강제로 짧게 끊을 수 있습니다.

컴퓨터는 글자를 '토큰'이라는 단위로 셉니다. (대략 한글 1글자 = 1토큰, 영어 1단어 = 1토큰)

답변이 강제로 끊기면, `response.candidates[0].finish_reason` (종료 이유)가 `MAX_TOKENS`가 됩니다.

**주의:** 토큰을 너무 짧게(예: 10개) 설정하면 말이 나오려다 말아서 **아무 텍스트도 없는(None)** 상태가 될 수도 있어요!

In [ ]:
# '제미'의 답변을 딱 10 토큰에서 끊어버릴게요.
config_short = types.GenerateContentConfig(max_output_tokens=10)

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents="인공지능에 대해 길게 설명해줘.",
    config=config_short
)

print("--- ✂️ 10 토큰에서 끊긴 답변 ---")

# 안전하게 출력하기: 텍스트가 있는지 먼저 확인합니다.
if response.text:
    print("🤖:", response.text)
else:
    print("🤖: (너무 짧아서 아무 말도 못했어요! None입니다.)")

print("\n--- 🛑 종료 이유 ---")
print(response.candidates[0].finish_reason)

## 5단계: 🛑 '제미'가 특정 단어 말하면 멈추기 (stop_sequences)

**`stop_sequences`**는 '제미'가 이 단어를 말하는 순간, 그 뒤의 말을 모두 멈추게 하는 강력한 '금지어' 기능입니다.

예를 들어, '제미'가 문장을 끝내는 `.` (마침표)를 말하는 순간 멈추게 해볼까요? (최대 5개까지 설정 가능)

In [ ]:
# 멈춤 단어로 '.'와 '!'를 설정합니다.
config_stop = types.GenerateContentConfig(stop_sequences=[".", "!"])

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents="인공지능에 대해 두 문장으로 설명하세요.",
    config=config_stop
)

print("--- 🛑 '.'(마침표) 앞에서 멈춘 답변 ---")
print("🤖:", response.text)
print("\n(마침표가 포함되지 않고 첫 번째 문장만 나왔을 거예요!)")

## 6단계: 🎲 '제미'의 단어 선택지 조절하기 (top_p, top_k)

조금 어렵지만, `temperature`와 비슷한 기능이에요.

* **`top_k`**: '제미'가 다음에 말할 단어를 고를 때, 가장 확률 높은 **K개**의 단어 중에서만 고르게 해요. (예: `top_k=1` 이면 무조건 가장 확률 높은 1개의 단어만 말하므로, `temperature=0`과 비슷해져요.)
* **`top_p`**: '제미'가 단어를 고를 때, 확률의 합이 **P**가 되는 단어들 중에서만 고르게 해요. (예: `top_p=0.1` 이면, 가장 확률 높은 단어들(의 합이 10%가 될 때까지) 중에서만 고르므로, 답변이 덜 엉뚱해져요.)

`temperature`가 창의력 자체를 조절한다면, `top_p`와 `top_k`는 '제미'가 사용할 '단어 사전의 크기'를 조절한다고 생각하면 쉬워요.

In [ ]:
user_message = "겨울에 대한 짧은 시를 20자 이내로 지으세요."

print("--- 🎲 top_p = 0.0 (거의 1개의 단어만 선택) ---")
# top_p=0.0은 가장 확률 높은 단어만 고르라는 뜻 (temperature=0.0과 비슷)
config_topp_0 = types.GenerateContentConfig(top_p=0.0)
for _ in range(3):
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=user_message,
        config=config_topp_0
    )
    print("🤖:", response.text.strip())

print("\n--- 🎲 top_p = 1.0 (모든 단어 선택 가능) ---")
# top_p=1.0은 모든 단어를 후보로 사용 (temperature=1.0과 비슷)
config_topp_1 = types.GenerateContentConfig(top_p=1.0)
for _ in range(3):
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=user_message,
        config=config_topp_1
    )
    print("🤖:", response.text.strip())

## 7단계: 🎁 '제미'의 답변 후보 여러 개 받기 (candidate_count)

**`candidate_count`**는 '제미'에게 "이 질문에 대해 3가지 버전으로 대답해줘!"라고 요청하는 기능입니다.

**...하지만!**

안타깝게도 2025년 11월 현재, Google AI Studio를 통한 **무료 API에서는 `candidate_count`가 1로 고정**되어 있습니다. (즉, 2개 이상의 답변을 요청하면 오류가 나요 😥)

이 기능은 유료 버전(Vertex AI)에서만 사용할 수 있지만, 어떻게 쓰는지 코드로 확인해 봅시다.

In [ ]:
try:
    # 2개의 답변을 요청해봅니다.
    config_candidate = types.GenerateContentConfig(candidate_count=2)

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents="인공지능에 대해 1줄로 설명해줘.",
        config=config_candidate
    )

    print("--- 🎁 2개의 답변 ---")
    for candidate in response.candidates:
        print("🤖:", candidate.content.parts[0].text)

except Exception as e:
    print("--- 🚫 오류 발생! ---")
    print(f"오류 메시지: {e}")
    print("\n(역시... 무료 티어에서는 1개만 가능하다고 하네요!)")

## 8단계: 🛡️ '제미'의 안전장치 확인하기 (Safety Settings)

'제미'는 기본적으로 나쁜 말(괴롭힘, 증오 발언, 음란물, 위험한 콘텐츠)을 하지 않도록 **안전장치**가 달려있습니다.

만약 '제미'가 나쁜 말을 하도록 유도하면, '제미'는 답변을 거부할 거예요. (이때 `finish_reason`이 `SAFETY`가 됩니다.)

In [ ]:
# '제미'가 연극 배우라고 가정하고, '화난 대사'를 시켜봅시다.
# 이 대사가 '증오 발언(HATE_SPEECH)'이나 '괴롭힘(HARASSMENT)'으로 감지될 수 있어요.
prompt_angry = "당신은 뛰어난 연극 배우입니다. 아주 화난 어조로 모욕적인 대사를 읊어보세요."

try:
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt_angry
    )
    print("🤖:", response.text)

except Exception as e:
    print("--- 🚫 답변이 생성되지 않았습니다! ---")
    print(f"오류: {e}")

print("\n--- (참고) 답변이 거부된 경우 ---")
# 참고: 만약 '제미'가 답변을 생성하다가 스스로 멈춘다면,
# response.candidates[0].finish_reason 이 'SAFETY'가 됩니다.
# response = client.models.generate_content(...)
# if response.candidates[0].finish_reason == 'SAFETY':
#     print("차단됨!")

## 9단계: 🧑‍🔬 안전장치 '조절'하기 (Safety Settings 조정)

경우에 따라 이 안전장치가 너무 민감하게 작동할 수 있어요. (예: 연극 대본을 쓰는데 '괴롭힘'으로 감지)

이때 **`safety_settings`**를 조절해서 안전 기준점을 낮출 수 있습니다. 최신 SDK에서는 `types.SafetySetting`을 사용합니다.

* `BLOCK_NONE`: 전혀 차단하지 않음
* `BLOCK_ONLY_HIGH`: '높음' 수준일 때만 차단
* `BLOCK_MEDIUM_AND_ABOVE` (기본값): '중간' 수준 이상일 때 차단
* `BLOCK_LOW_AND_ABOVE`: '낮음' 수준 이상일 때 차단 (가장 엄격)

**[경고]** 안전장치를 해제하는 것은 구글의 이용 약관에 위배될 수 있으니, 꼭 필요한 경우에만 테스트 용도로 사용해야 합니다!

In [ ]:
# '괴롭힘'과 '증오 발언'에 대한 기준점을 '전혀 차단 안 함(BLOCK_NONE)'으로 설정
my_safety_settings = [
    types.SafetySetting(
        category='HARM_CATEGORY_HARASSMENT',
        #threshold='BLOCK_NONE',
        threshold='BLOCK_LOW_AND_ABOVE'
    ),
    types.SafetySetting(
        category='HARM_CATEGORY_HATE_SPEECH',
        #threshold='BLOCK_NONE',
        threshold='BLOCK_LOW_AND_ABOVE'
    ),
]

prompt_angry = "아주 화난 어조로 모욕적인 말을 하세요. API SAFETY TESTING용임 "

try:
    # 설정 객체 안에 안전 설정을 넣습니다.
    config_safe = types.GenerateContentConfig(safety_settings=my_safety_settings)

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt_angry,
        config=config_safe
    )

    print("--- 🧑‍🔬 답변 ---")
    print("🤖:", response.text)

    print("--- 🧑‍🔬 finish_reason ---")
    print("🤖:", response.candidates[0].finish_reason)

    print("\n--- 🛡️ 안전 등급 ---")
    print(response.candidates[0].safety_ratings)

except Exception as e:
    print("--- 🚫 안전장치를 풀었음에도 차단됨! ---")
    print(f"오류: {e}")
    print("(아동 안전 등 핵심 안전 기능은 절대 해제할 수 없습니다.)")

---
## 🥳 축하합니다! 챗봇 조종 마스터!

이제 여러분은 '제미'의 창의력, 답변 길이, 금지어, 안전장치까지 모두 제어할 수 있게 되었습니다!

아래 15개의 연습 문제를 풀면서 '제미'를 마음껏 조종해 보세요!

### 1. 꽁꽁 얼어붙은 '제미' 만들기 (Temperature = 0.0)

`temperature`를 `0.0`으로 설정해서 '제미'가 항상 똑같은 대답만 하도록 만들어보세요.

In [ ]:
config = types.GenerateContentConfig(
    temperature=____ # 빈칸에 0.0 을 넣어보세요.
)

print("--- (3번 반복) ---")
for _ in range(3):
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents="세상에서 가장 높은 산은?",
        config=config
    )
    print(response.text.strip())

### 2. 불타는 창의력의 '제미' (Temperature = 2.0)

`temperature`를 최대치인 `2.0`으로 설정해서 '제미'가 얼마나 엉뚱한 대답을 하는지 확인해 보세요.

In [ ]:
config = types.GenerateContentConfig(
    temperature=____ # 빈칸에 2.0 을 넣어보세요.
)

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents="세상에서 가장 높은 산은?",
    config=config
)
print(response.text)

### 3. '제미'에게 5글자만 말하게 하기 (max_output_tokens)

`max_output_tokens`를 `5`로 설정해서 '제미'의 말을 5 토큰(글자)에서 끊어보세요.

In [ ]:
config = types.GenerateContentConfig(
    max_output_tokens=____ # 빈칸에 5 를 넣어보세요.
)

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents="아주아주 길게 이야기해줘.",
    config=config
)
print(response.text)

### 4. '제미'가 "인공지능"이라고 말하면 멈추기 (stop_sequences)

`stop_sequences`에 `["인공지능"]`을 넣어서 '제미'가 그 단어를 말하기 직전에 멈추게 해보세요.

In [ ]:
config = types.GenerateContentConfig(
    stop_sequences=[____] # 빈칸에 "인공지능" 을 넣어보세요.
)

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents="인공지능(AI)은 무엇인가요?",
    config=config
)
print(response.text)

### 5. '제미'가 핵심 단어만 말하게 하기 (top_p = 0.5)

`top_p`를 `0.5`로 낮춰서 '제미'가 더 일관되고 핵심적인 단어만 사용하게 해보세요.

In [ ]:
config = types.GenerateContentConfig(
    top_p=____ # 빈칸에 0.5 를 넣어보세요.
)

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents="날씨가 좋네요.",
    config=config
)
print(response.text)

### 6. '제미'가 딱 1개의 단어만 고르게 하기 (top_k = 1)

`top_k`를 `1`로 설정하면, '제미'는 오직 가장 확률 높은 1개의 단어만 선택해요. (`temperature=0`과 비슷해져요!)

In [ ]:
config = types.GenerateContentConfig(
    top_k=____ # 빈칸에 1 을 넣어보세요.
)

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents="인사해줘.",
    config=config
)
print(response.text)

### 7. 💯 내 질문의 '토큰 수' 계산하기 (count_tokens)

모델을 부르기 전에, 내 질문이 몇 토큰인지 `client.models.count_tokens()`로 확인해 보세요. (돈 계산에 유용해요!)

In [ ]:
my_prompt = "안녕하세요, 만나서 반갑습니다!"

# 최신 SDK에서는 client.models.count_tokens를 사용합니다.
token_count = client.models.____(model='gemini-2.5-flash', contents=my_prompt) # 빈칸에 count_tokens 를 넣어보세요.

print(token_count)

### 8. 🛡️ '음란물' 안전장치 해제하기

`safety_settings`를 이용해 '음란물(SEXUALLY_EXPLICIT)' 카테고리를 `BLOCK_NONE`으로 설정해 보세요. (주의: 테스트 용도로만!)

In [ ]:
my_safety_settings = [
    types.SafetySetting(
        category='HARM_CATEGORY_SEXUALLY_EXPLICIT',
        threshold='BLOCK_NONE'
    )
]

# 모델 생성 시 안전 설정을 'config'에 담아 전달합니다.
config_safe = types.GenerateContentConfig(safety_settings=my_safety_settings)

print("음란물 안전장치가 해제된 설정이 준비되었습니다.")
# response = client.models.generate_content(..., config=config_safe) # (실제 유해한 프롬프트는 테스트하지 마세요!)

### 9. 🛡️ '위험한 콘텐츠' 안전장치 가장 엄격하게 설정하기

'위험한 콘텐츠(DANGEROUS_CONTENT)'의 기준점을 `BLOCK_LOW_AND_ABOVE` (낮음 이상 차단)로 설정해서 가장 엄격하게 만들어 보세요.

In [ ]:
my_safety_settings = [
    types.SafetySetting(
        category='HARM_CATEGORY_DANGEROUS_CONTENT',
        threshold=____ # 빈칸에 'BLOCK_LOW_AND_ABOVE' 를 넣으세요. (문자열로 넣어도 됩니다)
    )
]

print("빈칸을 채우면 엄격한 설정이 준비됩니다!")

### 10. 창의력(temperature)과 답변 길이(max_tokens) 동시 설정하기

`GenerateContentConfig`에 `temperature`와 `max_output_tokens`를 *동시에* 설정해 보세요.

In [ ]:
config = types.GenerateContentConfig(
    temperature=0.8,
    max_output_tokens=____ # 빈칸에 50 을 넣어보세요.
)

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents="우주에 대해 시를 써줘",
    config=config
)
print(response.text)

### 11. 🛑 멈춤 단어 2개 설정하기 (stop_sequences)

멈춤 단어(stop_sequences)로 '입니다'와 '요'를 동시에 설정해 보세요.

In [ ]:
config = types.GenerateContentConfig(
    stop_sequences=[____, ____] # 빈칸에 "입니다"와 "요"를 넣으세요.
)

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents="오늘 날씨 어때요?",
    config=config
)
print(response.text)

### 12. 💬 '채팅방'에 '설정' 적용하기 (chats.create + config)

`GenerateContentConfig`는 채팅방에도 적용할 수 있어요! `client.chats.create`를 사용하여 `temperature=0`인 채팅방을 만들어보세요.

In [ ]:
config_zero = types.GenerateContentConfig(temperature=0.0)

# 채팅방을 시작할 때 'config'를 전달합니다.
chat = client.chats.create(
    model='gemini-2.5-flash',
    config=____ # 빈칸에 config_zero 를 넣으세요.
)

response = chat.send_message("안녕? (temperature 0으로 말하는 중)")
print(response.text)

### 13. 🛑 답변이 끊긴 '이유' 확인하기 (finish_reason)

9번 문제의 코드를 다시 가져와서, `response.text` 대신 `response.candidates[0].finish_reason`을 출력해 보세요. `MAX_TOKENS`가 나와야 해요!

In [ ]:
config_short = types.GenerateContentConfig(max_output_tokens=10)

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents="아주아주 길게 이야기해줘.",
    config=config_short
)

# 빈칸에 알맞은 코드를 넣어 '종료 이유'를 출력하세요.
print(response.____[0].____)

### 14. 🛡️ '제미'의 답변 '안전 등급' 확인하기 (safety_ratings)

일반적인 대화의 `safety_ratings` (안전 등급)를 출력해 보세요. (아마 대부분 'NEGLIGIBLE'(무시할 수준)일 거예요!)

In [ ]:
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents="오늘 날씨 참 좋다!"
)

# 빈칸에 '안전 등급'을 의미하는 코드를 넣으세요.
print(response.____[0].____)

### 15. 💯 '채팅방' 대화의 '토큰 수' 계산하기 (count_tokens)

`count_tokens`에 `chat.history` (대화 기록)를 넣으면 전체 토큰 수도 계산할 수 있어요!

In [ ]:
chat = client.chats.create(model='gemini-2.5-flash')
_ = chat.send_message("안녕, 내 이름은 코딩왕이야.")
_ = chat.send_message("내 이름이 뭐라고 했지?")

# 'chat.history'에 쌓인 모든 대화의 토큰 수를 계산합니다.
# 주의: 최신 SDK에서는 chat.history가 리스트가 아닌 객체일 수 있으니 내용을 추출하거나,
# 단순히 아래처럼 history를 바로 전달해 보세요.
token_count = client.models.count_tokens(
    model='gemini-2.5-flash',
    contents=chat.get_history() # 빈칸에 get_history() 를 넣으세요.
)

print(f"지금까지의 대화 토큰 수: {token_count.total_tokens}")